In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_theme()


In [ ]:
file_path = 'data/cleaned_student_performance.csv'
if not os.path.exists(file_path):
    file_path = 'P_2_StudentPerformanceFactors (1)(1).csv'

df = pd.read_csv(file_path)
df.head()


In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all').T)
display(df.isnull().sum().sort_values(ascending=False))
print('Duplicate rows:', df.duplicated().sum())


In [ ]:
if 'Exam_Score' in df.columns:
    invalid_scores = ~df['Exam_Score'].between(0, 100)
    print('Invalid Exam_Score rows:', invalid_scores.sum())
    df = df.loc[~invalid_scores].copy()

categorical_cols_all = df.select_dtypes(include='object').columns.tolist()
for col in categorical_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print('Remaining missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
outlier_summary = {}
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_summary[col] = ((df[col] < lower) | (df[col] > upper)).sum()

pd.Series(outlier_summary).sort_values(ascending=False).to_frame('Potential_Outliers')


In [ ]:
if {'Hours_Studied', 'Attendance'} <= set(df.columns):
    df['Study_Attendance_Index'] = df['Hours_Studied'] * df['Attendance'] / 100

df.head()


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Exam_Score'], kde=True)
plt.title('Exam Score Distribution')
plt.xlabel('Exam Score')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()


In [ ]:
if 'Parental_Involvement' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='Parental_Involvement', y='Exam_Score')
    plt.title('Exam Score by Parental Involvement')
    plt.tight_layout()
    plt.show()

if {'Hours_Studied', 'Exam_Score'} <= set(df.columns):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='Hours_Studied', y='Exam_Score', alpha=0.5)
    plt.title('Hours Studied vs Exam Score')
    plt.tight_layout()
    plt.show()


In [ ]:
corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
display(df['Exam_Score'].describe())
group_cols = ['Parental_Involvement', 'Access_to_Resources', 'Motivation_Level', 'Peer_Influence']
for col in group_cols:
    if col in df.columns:
        print('\n', col)
        display(df.groupby(col)['Exam_Score'].agg(['count', 'mean', 'median', 'std']).sort_values('mean', ascending=False))


In [ ]:
X = df.drop(columns=['Exam_Score'])
y = df['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)

    results.append({
        'Model': name,
        'Train_R2': r2_score(y_train, train_pred),
        'Test_MAE': mean_absolute_error(y_test, test_pred),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Test_R2': r2_score(y_test, test_pred)
    })
    trained_models[name] = pipe

model_results = pd.DataFrame(results).sort_values('Test_R2', ascending=False)
display(model_results)


In [ ]:
best_name = model_results.iloc[0]['Model']
best_model = trained_models[best_name]
best_pred = best_model.predict(X_test)
print('Best model:', best_name)
print('MAE:', round(mean_absolute_error(y_test, best_pred), 3))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, best_pred)), 3))
print('R2:', round(r2_score(y_test, best_pred), 3))


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=model_results, x='Test_R2', y='Model')
plt.title('Model Comparison by Test R²')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=best_pred, alpha=0.6)
plt.xlabel('Actual Exam Score')
plt.ylabel('Predicted Exam Score')
plt.title(f'Actual vs Predicted - {best_name}')
plt.tight_layout()
plt.show()


In [ ]:
os.makedirs('results', exist_ok=True)
os.makedirs('data', exist_ok=True)
model_results.to_csv('results/model_comparison.csv', index=False)
df.to_csv('data/cleaned_student_performance.csv', index=False)
print('Saved cleaned dataset and model comparison.')
